## 1 — Setup and imports

We import standard libraries, `sklearn` preprocessing utilities, and `joblib` for saving the pipeline.

In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import joblib

BASE = Path('data/outputs/grouped_imputed')
FILES = {
    '1Y': BASE / 'ensia_cleaned_1Y_grouped_imputed.csv',
    '2Y': BASE / 'ensia_cleaned_2Y_grouped_imputed.csv',
    '3Y': BASE / 'ensia_cleaned_3Y_grouped_imputed.csv',
}

def load_dataset(version='1Y'):
    p = FILES.get(version)
    if p is None or not p.exists():
        raise FileNotFoundError(f'Dataset for {version} not found: {p}')
    return pd.read_csv(p)

## 2 — Target selection and leakage protection

Choose the target (label) to predict. Defaults pick a reasonable overall average when available. To avoid data leakage we will remove other columns that contain aggregated averages (any column with 'avg'/'average'/'mean') before fitting the preprocessing pipeline.

In [ ]:
# Candidate target selection: prefer version-specific `<version>_average`
# choose dataset version and default candidate target (change before running)
SELECT_VERSION = '1Y'  # '1Y', '2Y', or '3Y'
TARGET_COLUMN = f"{SELECT_VERSION.lower()}_average"

def validate_target_in_df(df, target):
    if target in df.columns:
        return target
    cols_lower = {c.lower(): c for c in df.columns}
    if target.lower() in cols_lower:
        return cols_lower[target.lower()]
    # partial/fuzzy match: require all token parts to be in column name
    tparts = [p for p in re.split(r'[^a-z0-9]+', target.lower()) if p]
    for c in df.columns:
        cl = c.lower()
        if all(p in cl for p in tparts):
            return c
    return None

# Load the chosen dataset and pick a validated target (or auto-pick an overall average)
df = load_dataset(SELECT_VERSION)
validated = validate_target_in_df(df, TARGET_COLUMN)
if validated is None:
    # try to pick an overall / all-modules average automatically
    candidates = [c for c in df.columns if ('all' in c.lower() and ('avg' in c.lower() or 'average' in c.lower()))]
    if not candidates:
        candidates = [c for c in df.columns if ('average' in c.lower() or 'avg' in c.lower())]
    # prefer version-specific candidate
    version_key = SELECT_VERSION.lower()
    vcands = [c for c in candidates if version_key in c.lower()]
    validated = vcands[0] if vcands else (candidates[0] if candidates else None)

print('Selected dataset:', SELECT_VERSION)
print('Validated target column:', validated)
if validated is None:
    raise ValueError('Could not find a suitable target column automatically. Please set TARGET_COLUMN to an existing column name.')

## 3 — Rename columns to snake_case and shorten long names

We create a `clean_name()` function to normalize column names. We'll apply the renaming and save a cleaned CSV copy for the chosen version.

In [ ]:
def clean_name(name, max_len=40):
    s = str(name).strip()
    s = s.lower()
    s = s.replace(
, 
)
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_{2,}', '_', s)
    s = s.strip('_')
    if len(s) > max_len:
        parts = s.split('_')
        if len(parts) > 1:
            s = '_'.join(parts[:max_len//2] + parts[-(max_len//2):])
        s = s[:max_len]
    return s

def apply_rename_and_save(df, version=SELECT_VERSION, out_dir=Path('data/processed')):
    out_dir.mkdir(parents=True, exist_ok=True)
    rename_map = {c: clean_name(c) for c in df.columns}
    df2 = df.rename(columns=rename_map)
    out_path = out_dir / f'ensia_{version.lower()}_preprocessed.csv'
    df2.to_csv(out_path, index=False)
    return df2, out_path

df_clean, out_path = apply_rename_and_save(df, SELECT_VERSION)
'saved', out_path, df_clean.shape

In [ ]:
# Identify and remove leakage columns (other aggregated averages) before building preprocessor
target_col = validate_target_in_df(df_clean, TARGET_COLUMN) if TARGET_COLUMN else validated
if target_col is None:
    raise ValueError('Target column not found after renaming. Please set TARGET_COLUMN to an existing column name in the cleaned dataframe.')
# find other columns that look like aggregated averages/means and exclude them from features
leakage_cols = [c for c in df_clean.columns if (('avg' in c.lower() or 'average' in c.lower() or 'mean' in c.lower()) and c != target_col)]
print('Target column (cleaned):', target_col)
print('Detected leakage columns to drop:', leakage_cols)
# Create DataFrame used for fitting preprocessor (drop leakage cols and id-like identifiers)
df_for_preproc = df_clean.drop(columns=leakage_cols, errors='ignore')
df_for_preproc.shape

## 4 — Detect feature types and prepare ColumnTransformer

We automatically identify numeric columns and categorical columns. For categorical encoding strategy:
- `OneHotEncoder` for low-cardinality categorical features (<=10 unique values),
- `frequency encoding` for higher-cardinality object columns.
Numeric columns will be imputed (mean) and scaled with `StandardScaler`.

In [ ]:
def build_preprocessor(df, cat_ohe_thresh=10):
    id_like = [c for c in df.columns if re.search(r'id|index|record|student', c, re.IGNORECASE)]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in id_like]
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    ohe_cols = [c for c in obj_cols if df[c].nunique(dropna=False) <= cat_ohe_thresh]
    freq_cols = [c for c in obj_cols if c not in ohe_cols]

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])
    # handle OneHotEncoder API differences across sklearn versions
    try:
        ohe_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    except TypeError:
        ohe_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    ohe_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', ohe_encoder)
    ])
    preprocessor = ColumnTransformer(transformers=[
        ('num', num_pipe, numeric_cols),
        ('ohe', ohe_pipe, ohe_cols),
    ], remainder='drop')
    return preprocessor, numeric_cols, ohe_cols, freq_cols, id_like

preproc, numeric_cols, ohe_cols, freq_cols, id_like = build_preprocessor(df_for_preproc)
{'n_numeric': len(numeric_cols), 'n_ohe': len(ohe_cols), 'n_freq': len(freq_cols), 'id_like': id_like[:10]}

### Frequency encoding helper

For higher-cardinality categorical columns we perform frequency encoding (replace category by its relative frequency). This is simple, avoids high-dim one-hot vectors, and works well with many models.

In [ ]:
def apply_frequency_encoding(df, cols):
    df = df.copy()
    for c in cols:
        freq = df[c].value_counts(dropna=False, normalize=True)
        df[c + '_freq'] = df[c].map(freq).fillna(0.0)
        df = df.drop(columns=[c])
    return df

# demonstrate on a small sample
df_fe = apply_frequency_encoding(df_for_preproc, freq_cols[:3]) if freq_cols else df_for_preproc.copy()
'after_freq_shape', df_fe.shape

## 5 — Fit the preprocessor and save it

We will apply frequency encoding to the `freq_cols`, then fit the `ColumnTransformer` on the resulting DataFrame. Finally we save the fitted preprocessor using `joblib` so it can be reused in modeling notebooks.

In [ ]:
def fit_and_save_preprocessor(df, version=SELECT_VERSION, out_dir=Path('models')):
    out_dir.mkdir(parents=True, exist_ok=True)
    preproc, numeric_cols, ohe_cols, freq_cols, id_like = build_preprocessor(df)
    df2 = apply_frequency_encoding(df, freq_cols) if freq_cols else df.copy()
    df2 = df2.drop(columns=[c for c in id_like if c in df2.columns], errors='ignore')
    preproc.fit(df2)
    out_path = out_dir / f'preprocessor_{version.lower()}.joblib'
    joblib.dump({'preprocessor': preproc, 'freq_cols': freq_cols, 'id_like': id_like}, out_path)
    return out_path, preproc

out_path, preproc = fit_and_save_preprocessor(df_for_preproc, SELECT_VERSION)
'saved_preprocessor', out_path

In [ ]:
# Example: load and transform
obj = joblib.load(out_path)
pre = obj['preprocessor']
freq_cols = obj.get('freq_cols', [])
id_like = obj.get('id_like', [])
df_for_model = apply_frequency_encoding(df_for_preproc, freq_cols) if freq_cols else df_for_preproc.copy()
df_for_model = df_for_model.drop(columns=[c for c in id_like if c in df_for_model.columns], errors='ignore')
X = pre.transform(df_for_model)
'transformed_shape', X.shape

## Notes & next steps

- You can run the notebook for `2Y` and `3Y` by changing `SELECT_VERSION` at the top and re-running all cells.
- The code removes other aggregated average columns to avoid leakage; if you need a different leak rule, edit the detection cell.
- After you're satisfied, use the saved `preprocessor_{version}.joblib` in your modeling notebook to ensure consistent transforms.